In [106]:
from typing import Optional, List, Type
from pydantic import BaseModel, ValidationError, Field, validator
import pandas as pd
import geopandas as gpd
from shapely import wkb, Point
import numpy as np

**Define data reading and validation function**

In [85]:
# Define  Pydantic model
# Metro Fire Facilities
class Record(BaseModel):
    X: Optional[float] = Field(None, description="X coordinate")
    Y: Optional[float] = Field(None, description="Y coordinate")
    gnaf_confidence: Optional[float] = Field(None, ge=0, le=1, description="GNAF confidence score (0-1)")
    distance_to_gnaf: Optional[float] = Field(None, ge=0, description="Distance to GNAF point in meters")
    gnaf_lat: Optional[float] = Field(None, ge=-90, le=90, description="GNAF latitude")
    gnaf_long: Optional[float] = Field(None, ge=-180, le=180, description="GNAF longitude")
    geom: Optional[str] = Field(None, description="Geometry in WKT format")
    bpa_areaha: Optional[float] = Field(None, ge=0, description="Burned area in hectares")
    allcount: Optional[int] = Field(None, ge=0, description="Total count")
    yrsfrburn: Optional[int] = Field(None, ge=0, description="Years since last burn")
    firecount: Optional[int] = Field(None, ge=0, description="Fire count")
    burncount: Optional[int] = Field(None, ge=0, description="Burn count")
    zonetype: Optional[float] = Field(None, description="Zone type code")



def read_select(path: str, model_class: Type[BaseModel], usecols=None, filters=None, xy_cols=None, wkb_col=None, crs="EPSG:4326", encoding=None):
    encodings_to_try = ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252']
    if encoding:
        encodings_to_try = [encoding] + encodings_to_try

    df = None
    for enc in encodings_to_try:
        try:
            df = pd.read_csv(path, usecols=usecols + (list(filters.keys()) if filters else []), encoding=encoding)
            break
        except UnicodeDecodeError:
            continue

    if df is None:
        raise ValueError(f"Cannot read file: {path}")

    # Apply filters (on categorical fields)
    if filters:
        for col, val in filters.items():
            df = df[df[col] == val]
        # Drop filter columns after use
        df = df.drop(columns=list(filters.keys()), errors="ignore")

    df = df.dropna(subset=usecols)

    # Convert to Pydantic model
    records: List[BaseModel] = []
    for row in df.to_dict(orient="records"):
        filtered_row = {k: v for k, v in row.items() if k in model_class.model_fields}
        try:
            records.append(model_class(**filtered_row))
        except Exception as e:
            pass

    # Convert to GeoDataFrame if needed
    if xy_cols:
        gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df[xy_cols[0]], df[xy_cols[1]]), crs=crs)
    elif wkb_col:
        df["geometry"] = df[wkb_col].apply(lambda x: wkb.loads(bytes.fromhex(x), hex=True))
        gdf = gpd.GeoDataFrame(df.drop(columns=[wkb_col]), geometry="geometry", crs=crs)
    else:
        gdf = df  # just return the DataFrame if no geometry


    # keep columns
    columns = [c for c in df.columns if c != "geometry"]

    return records, gdf, columns

**Data readling and validation**

In [23]:
# Metro Fire Facilities
metro_records, metro_gdf,  metro_cols = read_select(
    "https://github.com/zhenweiw1/capstone/raw/refs/heads/main/Metro_Fire_Facilities.csv",
    model_class=Record,
    filters={"facility_operationalstatus": "OPERATIONAL", "facility_state": "VICTORIA"},
    usecols=["X","Y","gnaf_confidence","distance_to_gnaf","gnaf_lat","gnaf_long"],
    xy_cols=("X","Y")
)

In [26]:
rural_records, rural_gdf, rural_cols= read_select(
    "https://github.com/zhenweiw1/capstone/raw/refs/heads/main/Rural_Country_Fire_Service_Facilities.csv",
    model_class=Record,
    filters={"facility_operationalstatus": "OPERATIONAL", "facility_state": "VICTORIA"},
    usecols=["X","Y","gnaf_confidence","distance_to_gnaf","gnaf_lat","gnaf_long"],
    xy_cols=("X","Y")
)

In [27]:
bushfire_records, bushfire_gdf,bushfire_cols = read_select(
    "https://github.com/zhenweiw1/capstone/raw/refs/heads/main/vic_fire_bushfire_prone_area.csv",
    model_class=Record,
    usecols=["geom","bpa_areaha"],
    wkb_col="geom"
)

In [74]:
fire_history_records, fire_history_gdf, fire_history_cols  = read_select(
    "https://github.com/zhenweiw1/capstone/raw/refs/heads/main/vic_fire_history_latest.csv",
    model_class=Record,
    usecols=["geom","allcount","yrsfrburn","firecount","burncount"],
    wkb_col="geom"
)

In [88]:
fire_manage_records, fire_manage_gdf,fire_manage_cols = read_select(
    "https://github.com/zhenweiw1/capstone/raw/refs/heads/main/vic_fire_manage_zone.csv",
    model_class=Record,
    usecols=["geom","zonetype"],
    wkb_col="geom"
)

In [35]:
renewable_records, renewable_gdf, renewable_cols = read_select(
    "https://github.com/zhenweiw1/capstone/raw/refs/heads/main/renewable_project_nem.csv",
    model_class=Record,
    filters={"Region":"VIC1"},
    usecols=["X","Y"],
    xy_cols=("X","Y"),
    encoding='latin-1'
)

In [86]:
transmission_station_records, transmission_station_gdf, transmission_station_cols = read_select(
    "https://github.com/zhenweiw1/capstone/raw/refs/heads/main/transmission_substation.csv",
    model_class=Record,
    filters={"operationa": "Operational"},
    usecols=["geom"],
    wkb_col="geom"
)

**Training Data creation**

In [145]:
# first column is x
# second column is y
# third column is radius
# fouth column is total number of facilites
# fifth column is the closest facility

train_data=[]

for i in range(0,len(transmission_station_records)):
    x = transmission_station_gdf.iloc[i].geometry.x
    y = transmission_station_gdf.iloc[i].geometry.y
    location = np.array([x, y])

    radius=5
    closest_distance=10000

    number=0

    for j in range(0,10):
        facility = np.array([metro_gdf.iloc[j].geometry.x, metro_gdf.iloc[j].geometry.y])
        distance = np.linalg.norm(location - facility)

        if distance < radius:
            number+=1

        if distance < closest_distance:
            closest_distance = distance

    for j in range(0,10):
        facility = np.array([rural_gdf.iloc[j].geometry.x, rural_gdf.iloc[j].geometry.y])
        distance = np.linalg.norm(location - facility)

        if distance < radius:
            number+=1

        if distance < closest_distance:
            closest_distance = distance


    row = [x, y, radius, number, closest_distance]

    train_data.append(row)

train_data=np.array(train_data)

In [93]:
fire_manage_gdf.iloc[0].geometry.bounds

(141.17656858290073,
 -38.15355806861299,
 141.19931320500746,
 -38.14136343888582)

In [99]:
center = fire_manage_gdf.iloc[0].geometry.centroid
print(center.x, center.y)

141.1881735168308 -38.14701206611032
